In [1]:
import pandas as pd
import torch
from transformers import BitsAndBytesConfig, AutoModelForSequenceClassification, AutoTokenizer
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv('Kickstarter_processed.csv')


In [3]:
data["word_count"] = data["description"].fillna("").apply(lambda x: len(x.split()))

In [4]:
possible = data[data['word_count'] < 500]
possible

,Unnamed: 0,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,video,reached,status,duration,description_processed,pos_tagged,word_count
5,5,https://www.kickstarter.com/projects/ammcclend...,The Woman Who Knows: A Psychological Thriller ...,The crowdfunding campaign goal for The Woman W...,2136.0,2136.0,1850,10000.0,USD,Film & Video,1,21.360000,0,29,"['crowdfunding', 'woman', 'initially', 'raise'...","[('crowdfunding', 'VBG'), ('woman', 'NN'), ('i...",499
6,6,https://www.kickstarter.com/projects/sophiemar...,The Aughts,Mission Statement Return to the 2000s with The...,15240.0,15240.0,13201,15000.0,USD,Film & Video,1,101.600000,1,45,"['mission', 'statement', 'return', '2000s', 'h...","[('mission', 'NN'), ('statement', 'NN'), ('ret...",406
7,7,https://www.kickstarter.com/projects/themonkey...,Golden Goalie,Join Me Join me in a new community of Golden G...,316.0,316.0,273,15000.0,USD,Games,0,2.106667,0,30,"['join', 'join', 'community', 'golden', 'ice',...","[('join', 'NN'), ('join', 'NN'), ('community',...",349
11,11,https://www.kickstarter.com/projects/gamingcar...,Board Game Florida: 10 Events in 1 Year,WHAT WE DO OUR EVENTS Why This Kickstarter Exi...,16487.0,16487.0,14281,10000.0,USD,Games,1,164.870000,1,27,"['event', 'exist', 'standard', 'event', 'cost'...","[('event', 'NN'), ('exist', 'VB'), ('standard'...",199
15,15,https://www.kickstarter.com/projects/nightsea/...,NaN,The last year and a half was messy. Tired of b...,16034.0,16034.0,13888,13000.0,USD,Music,1,123.338462,1,33,"['last', 'half', 'messy', 'tired', 'decided', ...","[('last', 'JJ'), ('half', 'NN'), ('messy', 'NN...",338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7334,8375,https://www.kickstarter.com/projects/selfascop...,P.E.B. Pet Emergency Beacon (Patent Pending),*New Model P.E.B. Audio/Visual Beacon Top view...,822.0,822.0,704,10000.0,USD,Technology,1,8.220000,0,60,"['model', 'p', 'e', 'b', 'audio', 'visual', 'b...","[('model', 'NN'), ('p', 'NN'), ('e', 'NN'), ('...",359
7345,8386,https://www.kickstarter.com/projects/muxall/ir...,"IR Detector: Proximity Sensor for toys, roboti...",Muxall open source Infrared Detector (IRD) for...,1010.0,1010.0,865,9000.0,USD,Technology,1,11.222222,0,30,"['open', 'source', 'infrared', 'detector', 're...","[('open', 'JJ'), ('source', 'NN'), ('infrared'...",285
7346,8387,https://www.kickstarter.com/projects/137199743...,The Real Hovering Hoverboard - Personal Hoverc...,We're making real hoverboards a reality. After...,840.0,840.0,719,35000.0,USD,Technology,1,2.400000,0,23,"['real', 'reality', 'number', 'prototypes', 'e...","[('real', 'JJ'), ('reality', 'NN'), ('number',...",392
7347,8388,https://www.kickstarter.com/projects/360575576...,Camera Cuddlers-Smartphone Photography Made Ea...,I am so excited to have the opportunity to be ...,790.0,790.0,676,35000.0,USD,Technology,1,2.257143,0,50,"['excited', 'opportunity', 'excite', 'join', '...","[('excited', 'JJ'), ('opportunity', 'NN'), ('e...",169


In [5]:
possible = possible[possible['status'] == 0]
possible

,Unnamed: 0,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,video,reached,status,duration,description_processed,pos_tagged,word_count
5,5,https://www.kickstarter.com/projects/ammcclend...,The Woman Who Knows: A Psychological Thriller ...,The crowdfunding campaign goal for The Woman W...,2136.0,2136.0,1850,10000.0,USD,Film & Video,1,21.360000,0,29,"['crowdfunding', 'woman', 'initially', 'raise'...","[('crowdfunding', 'VBG'), ('woman', 'NN'), ('i...",499
7,7,https://www.kickstarter.com/projects/themonkey...,Golden Goalie,Join Me Join me in a new community of Golden G...,316.0,316.0,273,15000.0,USD,Games,0,2.106667,0,30,"['join', 'join', 'community', 'golden', 'ice',...","[('join', 'NN'), ('join', 'NN'), ('community',...",349
16,16,https://www.kickstarter.com/projects/230822529...,Jessenation - Recording My New Afrobeats Album,"Hi, I'm Jesse Emeghara. My stage name is Jesse...",20340.0,20340.0,17618,99000.0,USD,Music,1,20.545455,0,60,"['hi', 'jesse', 'stage', 'name', 'bear', 'nige...","[('hi', 'NN'), ('jesse', 'NN'), ('stage', 'NN'...",326
25,25,https://www.kickstarter.com/projects/148441575...,Computer Microchips Artistic Jewelry,I am Sergio Gutierrez. I am very passionate ab...,1044.0,1044.0,904,10000.0,USD,Technology,1,10.440000,0,30,"['sergio', 'gutierrez', 'passionate', 'art', '...","[('sergio', 'NN'), ('gutierrez', 'NN'), ('pass...",263
36,36,https://www.kickstarter.com/projects/makerleng...,MakerLength,What is MakerLength? Is a stop block build on ...,161.0,161.0,139,10000.0,USD,Technology,1,1.610000,0,30,"['stop', 'block', 'top', 'scale', 'basically',...","[('stop', 'JJ'), ('block', 'NN'), ('top', 'JJ'...",150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7334,8375,https://www.kickstarter.com/projects/selfascop...,P.E.B. Pet Emergency Beacon (Patent Pending),*New Model P.E.B. Audio/Visual Beacon Top view...,822.0,822.0,704,10000.0,USD,Technology,1,8.220000,0,60,"['model', 'p', 'e', 'b', 'audio', 'visual', 'b...","[('model', 'NN'), ('p', 'NN'), ('e', 'NN'), ('...",359
7345,8386,https://www.kickstarter.com/projects/muxall/ir...,"IR Detector: Proximity Sensor for toys, roboti...",Muxall open source Infrared Detector (IRD) for...,1010.0,1010.0,865,9000.0,USD,Technology,1,11.222222,0,30,"['open', 'source', 'infrared', 'detector', 're...","[('open', 'JJ'), ('source', 'NN'), ('infrared'...",285
7346,8387,https://www.kickstarter.com/projects/137199743...,The Real Hovering Hoverboard - Personal Hoverc...,We're making real hoverboards a reality. After...,840.0,840.0,719,35000.0,USD,Technology,1,2.400000,0,23,"['real', 'reality', 'number', 'prototypes', 'e...","[('real', 'JJ'), ('reality', 'NN'), ('number',...",392
7347,8388,https://www.kickstarter.com/projects/360575576...,Camera Cuddlers-Smartphone Photography Made Ea...,I am so excited to have the opportunity to be ...,790.0,790.0,676,35000.0,USD,Technology,1,2.257143,0,50,"['excited', 'opportunity', 'excite', 'join', '...","[('excited', 'JJ'), ('opportunity', 'NN'), ('e...",169


In [6]:
print(possible[possible['word_count'] == 200]['description'])
try_desc = possible['description'][2816]
print(possible['category'][2816])
try_desc
possible = possible.sample(n=min(5, len(possible)), random_state=67)
possible[['category', 'description', 'word_count']]

2816    The Aquatic Classroom Drowning is a health dis...
Name: description, dtype: object
Publishing


,category,description,word_count
1999,Film & Video,The Lego Spooder-Man Movie. In a Lego world fu...,280
4708,Film & Video,The Shift will be a new kind of t.v. show. A k...,236
1807,Games,Help Bring the Next Deck of 52 Prayer Cards to...,248
2519,Publishing,"Hello, and thank you for taking the time to ch...",360
1167,Film & Video,"This Halloween, you better wear anything but r...",496


In [34]:
pip install sentence_transformers

   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 571.3/571.3 kB 4.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [35]:
from transformers import AutoModel
from sklearn.metrics.pairwise import cosine_distances
import numpy as np
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2" # load the embedding model 
embed_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_model = SentenceTransformer(EMBED_MODEL_ID, device=embed_device)

def encode_texts(texts, batch_size=32, max_length=512): #turns list of text into embedding vectors
    text_list = ["" if pd.isna(text) else str(text) for text in texts]

    embed_model.max_seq_length = max_length

    embeddings = embed_model.encode(
        text_list,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True)
    
    return embeddings.astype('float32')

search_source = data.copy()
if "word_count" not in search_source.columns:
    search_source["word_count"] = search_source["description"].fillna("").astype(str).str.split().str.len()

successful = search_source[search_source["status"] == 1].copy()
successful_example_index = {}

for category_name, category_df in successful.groupby("category"):
    category_df = category_df.reset_index(drop=False).rename(columns={"index": "source_index"}).reset_index(drop=True)
    successful_example_index[category_name] = {
        "frame": category_df,
        "embeddings": encode_texts(category_df["description"].fillna("").astype(str).tolist()),
    }


def get_most_similar_successful_example(row, min_len_ratio=0.6, max_len_ratio=1.4, top_k=1):
    category_name = row["category"]
    if category_name not in successful_example_index:
        return pd.DataFrame()

    pool = successful_example_index[category_name]["frame"].copy()
    pool_embeddings = successful_example_index[category_name]["embeddings"]

    target_text = "" if pd.isna(row["description"]) else str(row["description"])
    target_length = len(target_text.split())

    length_mask = pool["word_count"].between(
        target_length * min_len_ratio,
        target_length * max_len_ratio,
    )

    if length_mask.any():
        pool = pool.loc[length_mask].reset_index(drop=True)
        pool_embeddings = pool_embeddings[length_mask.to_numpy()]

    target_embedding = encode_texts([target_text])
    distances = cosine_distances(target_embedding, pool_embeddings)[0]

    ranked = pool.assign(
        cosine_distance=distances,
        cosine_similarity=1 - distances,
    ).sort_values("cosine_distance", ascending=True)

    return ranked.head(top_k)


c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\simon\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 22/22 [00:09<00:00,  2.41it/s]


here we are loading a model: Meta-LLama 3.1 with 8billion parameters, the 'Instruct' version which is the base model but fine-tuned on instruction following, perfect for out task where the instruction is like 'rewrite this description'.
We cannot (probably, idk) load the whole model so we load the model but the parameters are quantized (which I guess means like transformed in a way so that they take up much less memory) and then loaded into the model, with other settings I commented here below.

In [36]:
# here we load the model but with the parameters quantized 
model_id  = 'mistralai/Mistral-7B-Instruct-v0.3'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=400, #limits the words (we need to put it also in the prompt I think)
        temperature=0.8, # we can see if different temperatures give different results also
        do_sample=True, #it says like its required for the temperature setting to work I guess
        repetition_penalty=1.03, #the higher = more penalization in repetitive output 
        return_full_text=False,
    ),
    model_kwargs={"quantization_config": quantization_config},
)

Loading weights:   1%|          | 2/291 [00:04<08:54,  1.85s/it]c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:34<00:00,  8.42it/s]


In [37]:
ROLE_MAP = {"human": "user", "ai": "assistant", "system": "system"}

def chatpromptvalue_to_olmo(pv):
    # IMPORTANT: use pv.to_messages() (not pv.to_string()) to avoid any default formatting
    msgs = pv.to_messages()
    chat = [{"role": ROLE_MAP[m.type], "content": m.content} for m in msgs]
    return llm.pipeline.tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

to_olmo = RunnableLambda(chatpromptvalue_to_olmo) #note: we need to modify the names 'to_olmo' since our model is not olmo


In [38]:
# The user prompt, we need to modify it based on what we wanna do with our task 
user_prompt = '''Rewrite the unsuccessful Kickstarter campaign in the category {category} 
using the successful example only as stylistic and structural guidance.

Do not copy phrases.
Do not invent factual details.
Preserve the original project idea.

Unsuccessful campaign:
{sentence}

Most similar successful campaign:
{example}

Rewritten version: ''' 

# The full prompt, in chat format. In this case, we are in the first
# conversation turn and therefore we only have the sytem and the user prompts.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", ""),
        ("human", user_prompt),
    ]
)

In [ ]:
for i in range(len(possible)):
    row = possible.iloc[i]
    example_df = get_most_similar_successful_example(row, top_k=1)

print(possible.iloc[0])
print(example_df['description'])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# -- Pipe it: prompt | (messages->string) | llm --
chain = prompt | to_olmo | llm | StrOutputParser()

def clean_response(text):
    text = text.strip()

    if "[/INST]" in text:
        text = text.split("[/INST]", 1)[-1].strip()

    if text.startswith("<s>"):
        text = text[3:].strip()

    if text.startswith("[INST]"):
        text = text[len("[INST]"):].strip()

    if "New description:" in text:
        text = text.split("New description:", 1)[-1].strip()

    if text.startswith("Rewritten version:"):
        text = text[len("Rewritten version:"):].strip()

    return text

generated_rows = []

for i in range(len(possible)):
    row = possible.iloc[i]
    example_df = get_most_similar_successful_example(row, top_k=1)
    example_text = example_df.iloc[0]["description"] if not example_df.empty else ""

    response = clean_response(chain.invoke({
        "sentence": row["description"],
        "category": row["category"],
        "example": example_text,
    }))
    generated_rows.append({
        "sentence": row["description"],
        "category": row["category"],
        "example": example_text,
        "response": response,
    })

possible_generated = pd.DataFrame(generated_rows)
possible_generated

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for 

,sentence,category,example,response
0,The Lego Spooder-Man Movie. In a Lego world fu...,Film & Video,"The Story The story begins on a wet, gloomy Oc...",Title: The Brick-tacular Spider-Dud: A Lego Ad...
1,The Shift will be a new kind of t.v. show. A k...,Film & Video,"Dear friends, family, and fellow movie-lovers,...","Dear friends, family, and fellow enthusiasts o..."
2,Help Bring the Next Deck of 52 Prayer Cards to...,Games,Story Pledge Levels Add-Ons Quick Overview (Th...,Unleash Faith-Empowering Playing Cards: The 52...
3,"Hello, and thank you for taking the time to ch...",Publishing,"Podcasting rules. I’m Dan Vonderheide, a Louis...",Greetings and thank you for considering my Kic...
4,"This Halloween, you better wear anything but r...",Film & Video,"The Story The story begins on a wet, gloomy Oc...",Title: The Crimson Veil\n\nBrace yourself for ...


In [31]:
classifier_dir = "saved_bert_model"
classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_dir)
classifier_model = AutoModelForSequenceClassification.from_pretrained(classifier_dir)
classifier_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier_model.to(classifier_device)
classifier_model.eval()

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10566.96it/s]


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, b

In [32]:
def classify_responses(df, text_col="response", batch_size=16, max_length=512):
    texts = df[text_col].fillna("").astype(str).tolist()
    predicted_labels = []
    success_probabilities = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        inputs = classifier_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {key: value.to(classifier_device) for key, value in inputs.items()}

        with torch.no_grad():
            logits = classifier_model(**inputs).logits
            probabilities = torch.softmax(logits, dim=-1)

        predicted_labels.extend(probabilities.argmax(dim=-1).cpu().tolist())
        success_probabilities.extend(probabilities[:, 1].cpu().tolist())

    scored = df.copy()
    scored["predicted_label"] = predicted_labels
    scored["predicted_outcome"] = scored["predicted_label"].map({0: "Not successful", 1: "Successful"})
    scored["success_probability"] = success_probabilities
    return scored

possible_generated = classify_responses(possible_generated, text_col="response")
possible_generated[["category", "response", "predicted_outcome", "success_probability"]]

,category,response,predicted_outcome,success_probability
0,Film & Video,Title: The Brick-tacular Spider-Dud: A Lego Ad...,Successful,0.591251
1,Film & Video,"Dear friends, family, and fellow enthusiasts o...",Successful,0.590426
2,Games,Unleash Faith-Empowering Playing Cards: The 52...,Successful,0.590423
3,Publishing,Greetings and thank you for considering my Kic...,Successful,0.592699
4,Film & Video,Title: The Crimson Veil\n\nBrace yourself for ...,Successful,0.571244


In [33]:
possible_generated["predicted_outcome"].value_counts()

predicted_outcome
Successful    5
Name: count, dtype: int64